## ERP Price Silver Ttransformation

### 00. Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col
from datetime import datetime

### 01. Build today's folder name and read from bronze table

In [0]:
today = datetime.today().strftime("%Y_%m_%d")   # e.g. 2026_06_23
bronze_table = f"`abc_business-core`.bronze_{today}.erp_pricing"
df = spark.table(bronze_table)

### 02. Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### 03. Normalize Maintenance Flag to Boolean

In [0]:
df = df.withColumn(
    "maintenance",
    F.when(F.upper(col("maintenance")) == "YES", F.lit(True))
     .when(F.upper(col("maintenance")) == "NO", F.lit(False))
     .otherwise(None)
)

### 04. Renaming Columns

In [0]:
RENAME_MAP = {
    "id": "category_id",
    "cat": "category",
    "subcat": "subcategory",
    "maintenance": "maintenance_flag"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

### 05. Sanity checks of dataframe

In [0]:
df.limit(5).display()

### 06. Writing Silver Table

In [0]:
table_name = f"`abc_business-core`.silver.erp_pricing_{today}"
df.write.mode("overwrite").format("delta").saveAsTable(table_name)

### 07. Sanity checks of silver table

In [0]:
df = spark.sql(f"SELECT * FROM {table_name} LIMIT 5")
df.show()